# WTI Crude Oil Price Forecasting — Stateless Methods: Systematic Backtest (Notebook 4 of 7)

This notebook simulates a rigorous production forecasting workflow:

1. Run a **rolling weekly backtest across 2025** using
   `energy_oil_backtest.yaml` for all candidate predictors.
2. Compute metrics — **CRPS** for 5/10/21-day trajectories.
3. Select the **top contender configurations** based solely on 2025
   historical performance (no peeking at 2026).
4. Let the contenders compete in the **2026 Protected Arena**
   (`energy_oil_eval.yaml`) during the geopolitical price shock —
   measuring adaptive real-time responsiveness and calibration.

The line-up spans three families behind one `Predictor` interface: **baselines**
(Naive, AutoARIMA), **numerical ML** (LightGBM ± a leak-safe covariate panel),
and **LLM/agent** methods (LLM-process forecasters and a news-reading analyst
agent) — the last run on *both* project models, `gemini-3.1-flash-lite-preview`
and `gemini-3.5-flash`. Every predictor is one toggle line in the registry in
Section 2. Agent configs come from `energy_oil_forecasting.analyst_agent`.

---
## 1. Setup, Data Registration & Spec Loading

In [1]:
import warnings
from pathlib import Path

import energy_oil_forecasting
import pandas as pd
import yaml
from aieng.forecasting.evaluation import (
    MultiTargetBacktestSpec,
    cached_multi_backtest,
    describe_spec,
)
from aieng.forecasting.models import ADVANCED_MODEL, LITE_MODEL
from energy_oil_forecasting.data import (
    DEFAULT_WTI_COVARIATE_SERIES_IDS,
    build_wti_multivariate_service,
)


warnings.filterwarnings("ignore")

# ── Mode ──────────────────────────────────────────────────────────────────────
# Set SMOKE_TEST = True to run a 2-origin, 1-sample version of the notebook
# for fast local development and end-to-end CI testing. The full specs run
# 51 backtest + 8 eval origins; smoke runs 2 + 2.
SMOKE_TEST = True

# ── Models ────────────────────────────────────────────────────────────────────
# The project standardises on two Vector-proxy models. Every LLM and agent
# predictor below is run once per model so we can compare them head-to-head.
# (bare proxy names — no "gemini/" prefix)
MODELS = [LITE_MODEL, ADVANCED_MODEL]  # "gemini-3.1-flash-lite-preview", "gemini-3.5-flash"

# ── Derived settings (do not edit below) ─────────────────────────────────────
N_SAMPLES = 1 if SMOKE_TEST else 3  # trajectories per LLMP-Sampled call

# LightGBM hyperparameters (shared by the univariate and +covariate variants).
LAGS = 21  # one trading month of lagged target/covariate history
NUM_SAMPLES_LGBM = 100 if SMOKE_TEST else 200  # Monte-Carlo draws for quantiles
LGBM_KWARGS = {"num_threads": 1, "n_jobs": 1, "verbosity": -1}  # deterministic, quiet

# Data service: WTI target + a leak-safe covariate panel (all Yahoo Finance —
# Brent, natural gas, gasoline, gold, USD index, the USL/USO futures-curve
# contango proxy, and VIX). Non-covariate predictors simply ignore the extras,
# so one service feeds the whole leaderboard. Unavailable tickers are skipped
# with a warning, so this still runs offline / under partial connectivity.
data_service = build_wti_multivariate_service()
COVARIATES = [c for c in DEFAULT_WTI_COVARIATE_SERIES_IDS if c in set(data_service.series_ids)]

spec_dir = Path(energy_oil_forecasting.__file__).parent / "specs"
if SMOKE_TEST:
    backtest_file, eval_file = "energy_oil_smoke.yaml", "energy_oil_eval_smoke.yaml"
else:
    backtest_file, eval_file = "energy_oil_backtest.yaml", "energy_oil_eval.yaml"

with open(spec_dir / backtest_file) as f:
    backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))
with open(spec_dir / eval_file) as f:
    eval_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

print(f"{'⚡ SMOKE MODE' if SMOKE_TEST else '📊 FULL MODE'} — MODELS={MODELS}  N_SAMPLES={N_SAMPLES}")
print(f"Covariates registered ({len(COVARIATES)}): {', '.join(COVARIATES) or '(none)'}")
print()
print("━" * 72)
print("LOADED SPECIFICATIONS:")
print("━" * 72)
print(describe_spec(backtest_spec, data_service))
print(describe_spec(eval_spec, data_service))

⚡ SMOKE MODE — MODELS=['gemini-3.1-flash-lite-preview', 'gemini-3.5-flash']  N_SAMPLES=1
Covariates registered (7): brent_log_ret_1b_l1b, natgas_log_ret_1b_l1b, gasoline_log_ret_1b_l1b, gold_log_ret_1b_l1b, dollar_index_log_ret_1b_l1b, oil_curve_contango_l1b, vix_level_l1b

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LOADED SPECIFICATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MultiTargetBacktestSpec (spec_id=energy_oil_smoke)
  description: Two-origin smoke backtest for local and CI testing of the NB04 pipeline. Uses the same tasks, horizons, and warmup as energy_oil_backtest but with only 2 weekly origins so the full notebook can be exercised without burning tokens on 51 × 5 predictor evaluations.
  start:       2025-06-02 00:00:00
  end:         2025-06-09 00:00:00
  stride:      5
  warmup:      250
  tasks:       1

Task: wti_oil_price_forecast
  description: WTI Crude Oil continuous front-month futures Close price (y

---
## 2. Candidate Predictors

This experiment puts a full slate of methods on the same `Predictor` interface and
the same rolling backtest, spanning three families:

| Family | Predictors | Role |
|---|---|---|
| **Baselines** | `Naive (Last Value)`, `AutoARIMA` | Carry-forward floor + the classical statistical anchor |
| **Numerical ML** | `LightGBM`, `LightGBM + cov` (+ optional `Prophet`) | Gradient-boosted quantile regression on lagged price (and a leak-safe covariate panel — Brent, gas, gasoline, gold, USD index, the futures-curve contango proxy, and VIX). LightGBM-with-covariates was the strongest method in the S&P 500 study. |
| **LLM / Agent** | `LLMP-Sampled`, `LLMP-Grid`, `News Agent` — each on **both** project models | LLM-process forecasters and a news-reading analyst agent, run on `gemini-3.1-flash-lite-preview` *and* `gemini-3.5-flash` |

The predictor cell below is a **registry**: every method is one line with an
`enabled` flag. Flip a flag to add or drop a predictor — the rest of the
notebook (backtest, scoring, eval, scorecard) iterates over whatever is active.
The two baselines are flagged `baseline=True` and are the only results written to
`adaptive_agent/curriculum/` for Notebooks 5–6, so toggling the others never
disturbs the downstream training data.

In [2]:
from dataclasses import dataclass
from typing import Callable

from aieng.forecasting.methods import (
    LastValuePredictor,
    QuantileGridLLMPredictor,
    QuantileGridLLMPredictorConfig,
    SampledTrajectoryLLMPredictor,
    SampledTrajectoryLLMPredictorConfig,
)
from aieng.forecasting.methods.numerical.darts_arima import DartsAutoARIMAPredictor
from aieng.forecasting.methods.numerical.darts_regression import DartsLightGBMPredictor
from energy_oil_forecasting.analyst_agent import build_wti_agent_predictor, build_wti_news_config
from energy_oil_forecasting.prophet_baseline import ProphetPredictor


@dataclass
class PredictorEntry:
    """One row in the experiment. Flip ``enabled`` to switch a predictor on/off."""

    name: str
    factory: Callable[[], object]  # lazy — built only when enabled
    enabled: bool = True
    baseline: bool = False  # baselines are saved to curriculum/ for NB05–06


# LLM / agent factories — each takes a model so the same recipe runs on both.
# LLMP-Sampled optionally serializes the covariate panel into the prompt
# (labeled exogenous-series blocks); the others are target-only. A distinct
# variant_tag keeps the +cov run separate in the cache and on the leaderboard.
def _llmp_sampled(model, covariates=None):
    return SampledTrajectoryLLMPredictor(
        SampledTrajectoryLLMPredictorConfig(
            model=model,
            n_samples=N_SAMPLES,
            covariate_series_ids=covariates,
            variant_tag="cov" if covariates else None,
        )
    )


def _llmp_grid(model):
    return QuantileGridLLMPredictor(QuantileGridLLMPredictorConfig(model=model))


def _news_agent(model):
    return build_wti_agent_predictor(build_wti_news_config(model=model))


# ── Experiment registry ───────────────────────────────────────────────────────
# Toggle `enabled` on any line to include/exclude that predictor. LLM and agent
# methods are listed once per model so each can be switched on/off individually.
REGISTRY = [
    # Baselines — always saved to curriculum/ for the adaptive-agent notebooks.
    PredictorEntry("Naive (Last Value)", LastValuePredictor, enabled=True, baseline=True),
    PredictorEntry("AutoARIMA", DartsAutoARIMAPredictor, enabled=True, baseline=True),
    # Numerical ML — LightGBM was the strongest method in the S&P 500 study.
    PredictorEntry(
        "LightGBM",
        lambda: DartsLightGBMPredictor(lags=LAGS, num_samples=NUM_SAMPLES_LGBM, lgbm_kwargs=LGBM_KWARGS),
        enabled=True,
    ),
    PredictorEntry(
        "LightGBM + cov",
        lambda: DartsLightGBMPredictor(
            lags=LAGS,
            lags_past_covariates=LAGS,
            covariate_series_ids=COVARIATES,
            num_samples=NUM_SAMPLES_LGBM,
            lgbm_kwargs=LGBM_KWARGS,
        ),
        enabled=True,
    ),
    PredictorEntry("Prophet", ProphetPredictor, enabled=False),
    # LLM processes and the news agent — one row per model in MODELS.
    PredictorEntry(f"LLMP-Sampled ({LITE_MODEL})", lambda: _llmp_sampled(LITE_MODEL), enabled=True),
    PredictorEntry(f"LLMP-Sampled ({ADVANCED_MODEL})", lambda: _llmp_sampled(ADVANCED_MODEL), enabled=True),
    # LLMP-Sampled with the covariate panel serialized into the prompt — the one
    # LLM method that can take covariates with no package change. Compare each of
    # these against its target-only twin above to see if context helps the LLM.
    PredictorEntry(f"LLMP-Sampled + cov ({LITE_MODEL})", lambda: _llmp_sampled(LITE_MODEL, COVARIATES), enabled=True),
    PredictorEntry(
        f"LLMP-Sampled + cov ({ADVANCED_MODEL})", lambda: _llmp_sampled(ADVANCED_MODEL, COVARIATES), enabled=True
    ),
    PredictorEntry(f"LLMP-Grid ({LITE_MODEL})", lambda: _llmp_grid(LITE_MODEL), enabled=True),
    PredictorEntry(f"LLMP-Grid ({ADVANCED_MODEL})", lambda: _llmp_grid(ADVANCED_MODEL), enabled=True),
    PredictorEntry(f"News Agent ({LITE_MODEL})", lambda: _news_agent(LITE_MODEL), enabled=True),
    PredictorEntry(f"News Agent ({ADVANCED_MODEL})", lambda: _news_agent(ADVANCED_MODEL), enabled=True),
]

# Instantiate only the enabled predictors (lazy factories skip the rest).
PREDICTORS = {e.name: e.factory() for e in REGISTRY if e.enabled}
_BASELINE_PREDICTORS = {e.name for e in REGISTRY if e.baseline}

print(f"Active predictors ({len(PREDICTORS)}):")
for name in PREDICTORS:
    tag = "  (baseline → curriculum/)" if name in _BASELINE_PREDICTORS else ""
    print(f"  {name}{tag}")

Active predictors (12):
  Naive (Last Value)  (baseline → curriculum/)
  AutoARIMA  (baseline → curriculum/)
  LightGBM
  LightGBM + cov
  LLMP-Sampled (gemini-3.1-flash-lite-preview)
  LLMP-Sampled (gemini-3.5-flash)
  LLMP-Sampled + cov (gemini-3.1-flash-lite-preview)
  LLMP-Sampled + cov (gemini-3.5-flash)
  LLMP-Grid (gemini-3.1-flash-lite-preview)
  LLMP-Grid (gemini-3.5-flash)
  News Agent (gemini-3.1-flash-lite-preview)
  News Agent (gemini-3.5-flash)


---
## 3. Run the 2025 Historical Backtest

All 51 weekly origins in 2025 are evaluated for each predictor.
`cached_multi_backtest` caches results under `data/predictions/` so
subsequent runs are instant.

In [3]:
print(f"Running 2025 rolling backtest ({len(PREDICTORS)} predictor(s))...")
print("LLM/agent runs are expensive — first run will take several minutes.\n")

backtest_results: dict[str, object] = {}
for _name, _predictor in PREDICTORS.items():
    backtest_results[_name] = cached_multi_backtest(_predictor, backtest_spec, data_service)
    print(f"  {_name} ✓")

print("\nAll 2025 backtests complete.")

Running 2025 rolling backtest (12 predictor(s))...
LLM/agent runs are expensive — first run will take several minutes.

  Naive (Last Value) ✓
  AutoARIMA ✓
  LightGBM ✓
  LightGBM + cov ✓
  LLMP-Sampled (gemini-3.1-flash-lite-preview) ✓
  LLMP-Sampled (gemini-3.5-flash) ✓
  LLMP-Sampled + cov (gemini-3.1-flash-lite-preview) ✓
  LLMP-Sampled + cov (gemini-3.5-flash) ✓
  LLMP-Grid (gemini-3.1-flash-lite-preview) ✓
  LLMP-Grid (gemini-3.5-flash) ✓
  News Agent (gemini-3.1-flash-lite-preview) ✓
  News Agent (gemini-3.5-flash) ✓

All 2025 backtests complete.


---
## 4. Performance Characterisation

We score every active predictor on the 2025 backtest data:
- **CRPS** (Continuous Ranked Probability Score) — sharpness + calibration combined
- **MAE at h=21d** — point forecast accuracy at the longest horizon

The leaderboard ranks the families against each other — how much structure the
numerical methods (AutoARIMA, LightGBM ± covariates) extract over the naive
floor, whether the covariate panel earns its keep, and how the LLM/agent methods
compare across the two models. Where each method wins and where it struggles in
2025 is exactly the material the adaptive agent learns from in Notebook 5.

In [4]:
import math

from energy_oil_forecasting.analysis import score_backtest_results


leaderboard_rows = []
for name, results in backtest_results.items():
    scores = score_backtest_results(results, data_service)
    leaderboard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS": scores.get("mean_crps", float("nan")),
            "MAE h=21d": scores.get("mae_h21", float("nan")),
        }
    )

df_leaderboard = pd.DataFrame(leaderboard_rows).set_index("Predictor")
df_leaderboard = df_leaderboard.sort_values("Mean CRPS")

print("━" * 72)
print("2025 HISTORICAL BACKTEST — PERFORMANCE SUMMARY:")
print("━" * 72)
print(df_leaderboard.to_string())

arima_crps = df_leaderboard.loc["AutoARIMA", "Mean CRPS"] if "AutoARIMA" in df_leaderboard.index else float("nan")
naive_crps = (
    df_leaderboard.loc["Naive (Last Value)", "Mean CRPS"]
    if "Naive (Last Value)" in df_leaderboard.index
    else float("nan")
)
if not math.isnan(arima_crps):
    print(
        f"\nAutoARIMA CRPS improvement over Naive: {naive_crps - arima_crps:.4f} ({(naive_crps - arima_crps) / naive_crps:.1%})"
    )

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2025 HISTORICAL BACKTEST — PERFORMANCE SUMMARY:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                    Mean CRPS  MAE h=21d
Predictor                                                               
News Agent (gemini-3.1-flash-lite-preview)           3.533305   4.978333
News Agent (gemini-3.5-flash)                        3.728843   5.176666
AutoARIMA                                            3.882644   5.699750
LLMP-Grid (gemini-3.1-flash-lite-preview)            4.095949   5.054999
LLMP-Sampled (gemini-3.5-flash)                      4.344999   4.344999
LLMP-Grid (gemini-3.5-flash)                         4.572644   6.661666
LightGBM                                             4.683217   6.412367
LLMP-Sampled (gemini-3.1-flash-lite-preview)         5.074999   5.074999
LightGBM + cov                                       5.089612   6.705429
LLM

In [5]:
# ── Save backtest results for NB05 / NB06 ────────────────────────────────────
# Only the baseline predictors (flagged in the registry above) are written to
# curriculum/ so that toggling the other predictors on/off does not change the
# files NB05 and NB06 depend on.
_CURRICULUM_DIR = Path("adaptive_agent/curriculum")
_CURRICULUM_DIR.mkdir(exist_ok=True)
for _name, _result_dict in backtest_results.items():
    if _name not in _BASELINE_PREDICTORS:
        continue
    _result = next(iter(_result_dict.values()))
    (_CURRICULUM_DIR / f"backtest_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in backtest_results)} backtest result(s) to {_CURRICULUM_DIR}/")

Saved 2 backtest result(s) to adaptive_agent/curriculum/


---
## 5. 2026 Evaluation — Held-Out Test Period

We run every active predictor on **8 weekly origins in early 2026**
(`energy_oil_eval.yaml`) — a period of major geopolitical volatility not seen
during the 2025 backtest.

This evaluation serves two purposes:
1. **Measure out-of-sample robustness** — do the 2025 edges (statistical,
   covariate, or LLM/agent) hold under a structural regime shift?
2. **Establish the stateless baseline** that the trained adaptive agents in
   Notebook 6 are compared against. The baseline predictors' results are saved
   to `adaptive_agent/curriculum/` for Notebooks 5 and 6 to load.

In [6]:
print("Running 2026 evaluation...")
eval_results: dict[str, object] = {}
for name, predictor in PREDICTORS.items():
    eval_results[name] = cached_multi_backtest(predictor, eval_spec, data_service)
    print(f"  {name} ✓")

print("\n2026 evaluation complete.")

Running 2026 evaluation...
  Naive (Last Value) ✓
  AutoARIMA ✓
  LightGBM ✓
  LightGBM + cov ✓
  LLMP-Sampled (gemini-3.1-flash-lite-preview) ✓
  LLMP-Sampled (gemini-3.5-flash) ✓
  LLMP-Sampled + cov (gemini-3.1-flash-lite-preview) ✓
  LLMP-Sampled + cov (gemini-3.5-flash) ✓
  LLMP-Grid (gemini-3.1-flash-lite-preview) ✓
  LLMP-Grid (gemini-3.5-flash) ✓
  News Agent (gemini-3.1-flash-lite-preview) ✓
  News Agent (gemini-3.5-flash) ✓

2026 evaluation complete.


In [7]:
# ── Save eval results for NB06 ───────────────────────────────────────────────
# Only baseline predictors are written so uncommenting optional predictors
# above does not add extra rows to the NB06 scorecard.
for _name, _result_dict in eval_results.items():
    if _name not in _BASELINE_PREDICTORS:
        continue
    _result = next(iter(_result_dict.values()))
    (_CURRICULUM_DIR / f"eval_{_name}.json").write_text(_result.model_dump_json(), encoding="utf-8")
print(f"Saved {sum(n in _BASELINE_PREDICTORS for n in eval_results)} eval result(s) to {_CURRICULUM_DIR}/")

Saved 2 eval result(s) to adaptive_agent/curriculum/


---
## 6. Scorecard

Out-of-sample performance of every active predictor on the 2026 eval period.
These numbers are the **stateless baseline** the adaptive agent variants must
beat in Notebook 6 to demonstrate that training added value.

In [8]:
from energy_oil_forecasting.analysis import score_backtest_results


scorecard_rows = []
for name in PREDICTORS:
    if name not in eval_results:
        continue
    scores = score_backtest_results(eval_results[name], data_service)
    scorecard_rows.append(
        {
            "Predictor": name,
            "Mean CRPS (2026)": scores.get("mean_crps", float("nan")),
            "MAE h=21d (2026)": scores.get("mae_h21", float("nan")),
            "80% CI Coverage": scores.get("coverage_80", float("nan")),
        }
    )

df_scorecard = pd.DataFrame(scorecard_rows).set_index("Predictor")
df_scorecard = df_scorecard.sort_values("Mean CRPS (2026)")

print("━" * 72)
print("2026 EVAL SCORECARD — STATELESS BASELINE:")
print("━" * 72)
print(df_scorecard.to_string())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2026 EVAL SCORECARD — STATELESS BASELINE:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                    Mean CRPS (2026)  MAE h=21d (2026)  80% CI Coverage
Predictor                                                                                              
AutoARIMA                                                   5.924538          8.086147             50.0
News Agent (gemini-3.1-flash-lite-preview)                  6.515680          7.827498             50.0
News Agent (gemini-3.5-flash)                               6.996486          8.889998             50.0
LightGBM                                                    7.236753          8.107720             50.0
LLMP-Sampled + cov (gemini-3.1-flash-lite-preview)          7.269998          7.269998              0.0
LLMP-Sampled (gemini-3.5-flash)                             7.597498          7.5974

---
## 7. Diagnostics — reading past the leaderboard

The scorecard above is a single number per method. That hides *where* the score
comes from and *whether the ranking is even real*. The next cells decompose it
straight from the eval predictions — so they recompute on any rerun, smoke or
full:

- **CRPS by horizon** — does a method win everywhere, or is its mean dominated by
  one horizon? (For a short forecast, the 5-day calls are easy and nearly tied;
  the ranking is usually decided by the longest horizon.)
- **Mean CRPS ± standard error** — with only a handful of origins, are the gaps
  between methods bigger than the noise, or is the "winner" a coin flip?

With the **smoke spec (2 origins → a few scored points)** expect wide error bars
and an unstable ranking. That is exactly why a surprising leaderboard here is not
yet evidence of anything — it is a pipeline check.

In [9]:
from IPython.display import HTML, Markdown, display

from energy_oil_forecasting import viz
from energy_oil_forecasting.analysis import (
    build_price_frame,
    eval_narrative_md,
    extract_agent_rationales,
    leaderboard_with_uncertainty,
    per_horizon_crps,
    predictions_to_frame,
)


# Explode every scored 2026 eval prediction into one tidy row per
# (predictor, origin, horizon): point, 80% interval, realised price, and CRPS.
# Everything in Sections 7–10 reads from this frame, so it all recomputes when
# you flip SMOKE_TEST off and rerun.
price_df = build_price_frame(data_service)
eval_frame = predictions_to_frame(eval_results, data_service)
eval_board = leaderboard_with_uncertainty(eval_frame)
ph_crps = per_horizon_crps(eval_frame)

print("━" * 72)
print("MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):")
print("━" * 72)
print(ph_crps.round(2).to_string())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MEAN CRPS BY PREDICTOR × HORIZON (lower = better; 'All' = overall mean):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                                    h=5d  h=10d  h=21d    All
predictor                                                                    
AutoARIMA                                           1.20   2.15  10.18   5.92
News Agent (gemini-3.1-flash-lite-preview)          0.56   1.11  12.20   6.52
News Agent (gemini-3.5-flash)                       0.71   1.80  12.74   7.00
LightGBM                                            0.64   0.98  13.66   7.24
LLMP-Sampled + cov (gemini-3.1-flash-lite-preview)  1.65   2.89  12.27   7.27
LLMP-Sampled (gemini-3.5-flash)                     0.44   0.19  14.88   7.60
LLMP-Grid (gemini-3.1-flash-lite-preview)           0.79   3.39  13.16   7.62
LLMP-Grid (gemini-3.5-flash)                        0.63   2.54  14.58   8.08
L

In [10]:
# Heatmap of the table above. Read it left-to-right: the short-horizon columns
# are usually a near-uniform green (everyone is right), and one long-horizon
# column carries the colour spread that sets the 'All' ranking.
viz.make_crps_heatmap(ph_crps)

In [11]:
# Same leaderboard, now with a standard-error bar on each mean. If the bars of
# the top methods overlap, their ordering is not statistically distinguishable —
# the honest verdict when only a few origins have been scored.
viz.make_leaderboard_interval_chart(eval_board)

---
## 8. What are the top methods actually forecasting?

A CRPS number doesn't show *behaviour*. Below, each leading method's **median
forecast and 80% interval** are drawn against the realised WTI path at every
eval origin. This is where the leaderboard becomes legible — watch for who
tracks the move, who simply anchors to the last price, and whose intervals are
too narrow to cover the outcome when the market jumps.

In [12]:
# Plot the leaderboard's top methods, and always include the best LLM/agent
# method for contrast (so the chart compares families even when a baseline leads).
_leaders = list(eval_board.index[:3])
_best_llm = next((p for p in eval_board.index if eval_board.loc[p, "family"] == "LLM / Agent"), None)
if _best_llm and _best_llm not in _leaders:
    _leaders.append(_best_llm)
print(f"Showing: {', '.join(_leaders)}")
viz.make_eval_forecast_chart(eval_frame, price_df, _leaders)

Showing: AutoARIMA, News Agent (gemini-3.1-flash-lite-preview), News Agent (gemini-3.5-flash)


---
## 9. Reading the agent's reasoning

The news-reading agent attaches a free-text **rationale** to every forecast, and
a link to the full **Langfuse trace**. These are pulled straight from the
prediction metadata. This is where a surprising score becomes interpretable: you
can read whether the agent actually saw the geopolitical risk, and *how* it
turned that into a price and an interval — including, often, an interval far too
narrow for a regime shift.

In [13]:
# One card per (agent, origin): the rationale, the per-horizon note, and a link
# to the full reasoning trace. Empty only if no LLM/agent predictor is enabled.
eval_rationales = extract_agent_rationales(eval_results)
display(HTML(viz.render_rationales_html(eval_rationales)))

---
## 10. Takeaways — computed from this run

The summary below is **generated from the eval results in memory, not
hard-coded**, so it always matches what actually ran: the real winner, whether
its lead clears the noise floor, the horizon that decided the ranking, the
best-performing family, and a calibration line. Flip `SMOKE_TEST` off, rerun,
and these takeaways update themselves with the full leaderboard.

In [14]:
display(Markdown(eval_narrative_md(eval_frame, smoke=SMOKE_TEST)))

1. **AutoARIMA** has the best mean CRPS (5.92) on the 2026 evaluation, ahead of **News Agent (gemini-3.1-flash-lite-preview)** (6.52) by 0.59 — **well within the combined standard error**, so the ranking here is not statistically distinguishable from noise.
2. The leaderboard is **decided at h=21d**, where CRPS ranges 4.8–24.3 across methods; at the short h=5d horizon the methods are nearly tied (range 0.4–2.4). A handful of long-horizon points drives the whole ranking.
3. **By family** (mean CRPS): Numerical ML 7.11, LLM / Agent 7.90, Baseline 8.21. Best family this window: **Numerical ML**.
4. **Calibration:** AutoARIMA's 80% interval covered 50% of outcomes (target 80%) over its 4 scored point(s). With this few, coverage this far from target is itself a small-sample artefact, not necessarily mis-calibration.
5. ⚠️ **Smoke run:** only 2 origin(s) / 48 scored points. Treat the ranking as a pipeline check, not evidence — rerun the full suite before drawing conclusions.

---
## 11. What stateless methods can't do

Sections 7–10 score and dissect this run on its own terms. But every method here
shares one structural limit, independent of who topped the leaderboard: it is
calibrated (or prompted) **once and never updated between rounds**. That is
intentional — it creates a clean baseline — but it leaves a systematic gap:

- **No error feedback.** If a method's intervals are consistently too narrow in
  an elevated-vol regime (read the coverage line in Section 10, and the squashed
  error bars in Section 8), it keeps making the same mistake. Nothing updates its
  calibration between origins.

- **No strategy evolution.** Each prediction starts from the same prior — the
  same fitted model, or the same prompt. Resolved outcomes disappear without
  influencing future forecasts.

- **Context without memory.** Even the news agent re-reads the world each origin;
  it does not accumulate what worked. The rationales in Section 9 are written
  fresh every time, with no record of how the last one resolved.

→ **Notebook 5** introduces adaptive agents that study the 2025 backtest, record
systematic observations, and calibrate their strategies accordingly. At inference
time, each agent receives the live stateless estimate and decides how to adjust
it — applying what it learned from training.

→ **Notebook 6** evaluates whether any training approach actually improved
out-of-sample performance on the held-out 2026 data — measured against the
stateless baseline this notebook just established.